# Notebook 1: Database Setup & Pantry Initialization

## Purpose
This notebook sets up the SQLite pantry database, loads the 50-item JSON inventory,
and provides helper functions used by all other notebooks.

## Run this notebook first before any other notebook.

---

## 1.2 Imports

In [20]:
import mysql.connector
import json
import os
import sql_openai_config
from datetime import date, datetime
from pathlib import Path

# Paths — JSON file still used for seeding (optional now)
PROJECT_ROOT = Path(os.getcwd()).parent  # smart_pantry/
JSON_PATH = PROJECT_ROOT / 'data' / 'pantry_items.json'

MYSQL_CONFIG = sql_openai_config.get_mysql_config()


print(f"Project root : {PROJECT_ROOT}")
print(f"JSON path    : {JSON_PATH}")
print(f"MySQL DB     : {MYSQL_CONFIG['database']}")


Project root : c:\Users\sharm\OneDrive\Documents\DSS770_Prompt_Eng\smart_pantry
JSON path    : c:\Users\sharm\OneDrive\Documents\DSS770_Prompt_Eng\smart_pantry\data\pantry_items.json
MySQL DB     : smart_pantry


## 1.3 Database connection helper

In [21]:
import mysql.connector
from mysql.connector import Error

def get_connection():
    return mysql.connector.connect(
        host="localhost",
        user="pantry_user",
        password="MyStrongPwd123!",
        database="smart_pantry"
    )

def test_connection():
    try:
        conn = get_connection()
        if conn.is_connected():
            db_info = conn.get_server_info()
            print("✅ Connection successful!")
            print(f"Connected to MySQL Server version: {db_info}")

            cursor = conn.cursor()
            cursor.execute("SELECT DATABASE();")
            record = cursor.fetchone()
            print(f"Connected to database: {record[0]}")

    except Error as e:
        print("❌ Connection failed:")
        print(e)

    finally:
        if 'conn' in locals() and conn.is_connected():
            cursor.close()
            conn.close()
            print("🔒 MySQL connection closed.")

# Run the test
test_connection()

✅ Connection successful!
Connected to MySQL Server version: 8.0.45
Connected to database: smart_pantry
🔒 MySQL connection closed.


C:\Users\sharm\AppData\Local\Temp\ipykernel_14244\3447238679.py:16: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  db_info = conn.get_server_info()


In [22]:
def get_connection():
    """Return a MySQL connection to the pantry database."""
    return mysql.connector.connect(**MYSQL_CONFIG)

print("MySQL connection helper defined.")


MySQL connection helper defined.


## 1.4 Create the pantry table

The schema stores each pantry item with:
- `name` and `category` for identification
- `quantity` and `unit` for tracking how much is left
- `expiry_date` (ISO format) — the key field the agent reasons about
- `added_date` for auditing

In [23]:
def initialize_db():
    """Table already created in MySQL Workbench; nothing to do here."""
    print("Pantry table assumed to exist in MySQL (smart_pantry.pantry).")

initialize_db()


Pantry table assumed to exist in MySQL (smart_pantry.pantry).


## 1.5 Verify the loaded data

In [24]:
def show_pantry_table():
    """Display the full pantry table with days-until-expiry."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT name, category, quantity, unit, expiry_date FROM pantry ORDER BY expiry_date ASC"
    )
    rows = cursor.fetchall()
    conn.close()

    today = date.today()
    print(f"{'#':<4} {'Name':<20} {'Category':<12} {'Qty':>6} {'Unit':<8} {'Expiry':<12} {'Days left':>9}")
    print("-" * 78)
    for i, (name, cat, qty, unit, expiry) in enumerate(rows, 1):
        days = ""
        if expiry:
            d = expiry if isinstance(expiry, date) else datetime.strptime(str(expiry), "%Y-%m-%d").date()
            days = f"{(d - today).days:+d}"
        print(f"{i:<4} {name:<20} {cat:<12} {qty:>6} {unit:<8} {expiry:<12} {days:>9}")


In [25]:
show_pantry_table()


ProgrammingError: 1045 (28000): Access denied for user 'pantry_user'@'localhost' (using password: YES)

## 1.6 Quick stats — items by category and expiry urgency

✅ Connection successful!
Connected to MySQL Server version: 8.0.45
Connected to database: smart_pantry
🔒 MySQL connection closed.


C:\Users\sharm\AppData\Local\Temp\ipykernel_14244\3447238679.py:16: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  db_info = conn.get_server_info()


In [ ]:
def pantry_stats():
    """Print a summary of pantry health."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT name, expiry_date FROM pantry"
    )
    rows = cursor.fetchall()
    conn.close()

    today  = date.today()
    expired = []
    urgent  = []  # 0–3 days
    soon    = []  # 4–7 days
    fine    = []  # 8+ days

    for name, expiry in rows:
        if expiry:
            d = expiry if isinstance(expiry, date) else datetime.strptime(str(expiry), "%Y-%m-%d").date()
            days = (d - today).days
            if days < 0:
                expired.append(name)
            elif days <= 3:
                urgent.append(name)
            elif days <= 7:
                soon.append(name)
            else:
                fine.append(name)

    print('=== Pantry Health Summary ===')
    print(f' Total items : {len(rows)}')
    print(f' EXPIRED     : {len(expired)} → {expired}')
    print(f' AT RISK (0–3d): {len(urgent)} → {urgent}')
    print(f' SOON (4–7d) : {len(soon)} → {soon}')
    print(f' OK (8d+)    : {len(fine)}')


In [ ]:
pantry_stats()

=== Pantry Health Summary ===
 Total items : 352
 EXPIRED     : 106 → ['milk', 'tomato', 'eggs', 'butter', 'yogurt', 'cream', 'paneer', 'chicken breast', 'chicken thighs', 'ground beef', 'pork chops', 'salmon fillet', 'tilapia fillets', 'tofu firm', 'tempeh', 'broccoli', 'cauliflower', 'spinach fresh', 'kale', 'lettuce romaine', 'bell pepper red', 'bell pepper green', 'bell pepper yellow', 'cucumber', 'zucchini', 'eggplant', 'mushrooms button', 'mushrooms cremini', 'banana', 'orange', 'lime', 'grapes', 'strawberries', 'blueberries', 'raspberries', 'pineapple', 'mango', 'avocado', 'pear', 'kiwi', 'bread whole wheat', 'bread sourdough', 'burger buns', 'tortillas wheat', 'tortillas corn', 'naan', 'pita bread', 'bagels', 'croissants', 'wraps spinach', 'wraps tomato basil', 'orange juice', 'apple juice', 'milk', 'tomato', 'eggs', 'butter', 'yogurt', 'cream', 'paneer', 'chicken breast', 'chicken thighs', 'ground beef', 'pork chops', 'salmon fillet', 'tilapia fillets', 'tofu firm', 'tempeh', 

## 1.7 Reset helper (re-run if you want a clean slate)

In [ ]:
def reset_and_reload():
    """Drop all pantry rows and reload from JSON. Useful between experiments."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT ... FROM pantry ...")
    rows = cursor.fetchall()
    conn.close()

    load_from_json(JSON_PATH, clear_existing=False)
    print('Pantry reset and reloaded.')

# Uncomment to run:
# reset_and_reload()